# Multi-Ticker Momentum Backtest
**Strategy:** SMA20 + RSI(14) + Volume filter  
**Data source:** Alpaca Markets API (free account)  
**Author:** Algorithmic Claude Trading Bot — Day 2

### How to use
1. Enter your Alpaca paper API keys in Cell 2
2. Customize tickers, capital, and strategy params in Cell 3
3. Runtime → Run all cells


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install requests pandas numpy matplotlib seaborn tabulate -q

In [ ]:
# ── Cell 2: YOUR ALPACA CREDENTIALS ───────────────────────────────────────────
API_KEY    = "PASTE_YOUR_KEY_HERE"
API_SECRET = "PASTE_YOUR_SECRET_HERE"

# Base URLs
DATA_URL  = "https://data.alpaca.markets"
PAPER_URL = "https://paper-api.alpaca.markets"

HEADERS = {
    "APCA-API-KEY-ID": API_KEY,
    "APCA-API-SECRET-KEY": API_SECRET
}

In [ ]:
# ── Cell 3: STRATEGY PARAMETERS (customize here) ──────────────────────────────
TICKERS       = ["NVDA", "AAPL", "AMD", "TSLA", "META", "MSFT", "SPY"]
LOOKBACK_DAYS = 90        # how many trading days to backtest
STARTING_CAP  = 5000      # starting capital per ticker (isolated comparison)
POSITION_SIZE = 3000      # dollars deployed per trade
STOP_LOSS_PCT = 0.04      # 4% stop loss
RSI_BUY_MIN   = 45        # RSI lower bound for entry
RSI_BUY_MAX   = 65        # RSI upper bound for entry
RSI_SELL      = 75        # RSI overbought exit
SMA_PERIOD    = 20        # moving average window
RSI_PERIOD    = 14        # RSI window
VOL_PERIOD    = 20        # volume average window

In [ ]:
# ── Cell 4: Data fetching ──────────────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

def fetch_bars(symbol, days=90):
    end   = datetime.now().strftime("%Y-%m-%d")
    start = (datetime.now() - timedelta(days=days + 40)).strftime("%Y-%m-%d")
    url   = f"{DATA_URL}/v2/stocks/{symbol}/bars"
    params = {
        "timeframe": "1Day",
        "start": start,
        "end": end,
        "limit": days + 40,
        "adjustment": "split"
    }
    r = requests.get(url, headers=HEADERS, params=params)
    if r.status_code != 200:
        print(f"  ERROR {symbol}: HTTP {r.status_code} — {r.text[:200]}")
        return None
    bars = r.json().get("bars", [])
    if not bars:
        print(f"  ERROR {symbol}: No bars returned")
        return None
    df = pd.DataFrame(bars)
    df["date"] = pd.to_datetime(df["t"]).dt.date
    df = df.rename(columns={"o":"open","h":"high","l":"low","c":"close","v":"volume"})
    df = df[["date","open","high","low","close","volume"]].tail(days).reset_index(drop=True)
    print(f"  {symbol}: {len(df)} bars fetched ({df['date'].iloc[0]} → {df['date'].iloc[-1]})")
    return df

print("Fetching historical bars from Alpaca...")
raw_data = {}
for t in TICKERS:
    raw_data[t] = fetch_bars(t, LOOKBACK_DAYS)
print("\nFetch complete.")

In [ ]:
# ── Cell 5: Indicator calculation ─────────────────────────────────────────────
def calc_indicators(df):
    df = df.copy()
    # SMA
    df["sma"] = df["close"].rolling(SMA_PERIOD).mean()
    # RSI
    delta  = df["close"].diff()
    gain   = delta.clip(lower=0).rolling(RSI_PERIOD).mean()
    loss   = (-delta.clip(upper=0)).rolling(RSI_PERIOD).mean()
    rs     = gain / loss.replace(0, np.nan)
    df["rsi"] = 100 - (100 / (1 + rs))
    # Volume ratio
    df["vol_avg"] = df["volume"].rolling(VOL_PERIOD).mean()
    df["vol_ratio"] = df["volume"] / df["vol_avg"]
    return df

processed = {}
for t, df in raw_data.items():
    if df is not None:
        processed[t] = calc_indicators(df)

print(f"Indicators calculated for: {list(processed.keys())}")

In [ ]:
# ── Cell 6: Backtest engine ────────────────────────────────────────────────────
def backtest(df, symbol):
    cash      = STARTING_CAP
    position  = None
    trades    = []
    equity    = [STARTING_CAP]

    for i in range(SMA_PERIOD + RSI_PERIOD, len(df) - 1):
        row      = df.iloc[i]
        next_row = df.iloc[i + 1]
        price    = row["close"]
        next_open= next_row["open"]

        if pd.isna(row["rsi"]) or pd.isna(row["sma"]) or pd.isna(row["vol_ratio"]):
            equity.append(cash if not position else cash + position["shares"] * price)
            continue

        # ── EXIT logic ────────────────────────────────────────────────────────
        if position:
            drawdown   = (position["entry"] - price) / position["entry"]
            exit_reason = None
            if row["rsi"] > RSI_SELL:
                exit_reason = f"RSI {row['rsi']:.0f} > {RSI_SELL}"
            elif drawdown >= STOP_LOSS_PCT:
                exit_reason = f"Stop loss {drawdown*100:.1f}%"
            if exit_reason:
                proceeds = position["shares"] * next_open
                pnl      = proceeds - position["cost"]
                cash    += proceeds
                trades.append({
                    "ticker": symbol, "date": str(next_row["date"]),
                    "action": "SELL", "price": round(next_open, 2),
                    "shares": round(position["shares"], 4),
                    "pnl": round(pnl, 2), "reason": exit_reason,
                    "capital": round(cash, 2)
                })
                position = None

        # ── ENTRY logic ───────────────────────────────────────────────────────
        elif (price > row["sma"] and
              RSI_BUY_MIN <= row["rsi"] <= RSI_BUY_MAX and
              row["vol_ratio"] > 1.0):
            spend  = min(POSITION_SIZE, cash)
            if spend > 50:
                shares = spend / next_open
                cash  -= spend
                position = {"entry": next_open, "shares": shares, "cost": spend}
                trades.append({
                    "ticker": symbol, "date": str(next_row["date"]),
                    "action": "BUY",  "price": round(next_open, 2),
                    "shares": round(shares, 4),
                    "pnl": None,
                    "reason": f"RSI {row['rsi']:.0f}, price > SMA{SMA_PERIOD}, vol {row['vol_ratio']:.2f}x",
                    "capital": round(cash, 2)
                })

        current_val = cash + (position["shares"] * price if position else 0)
        equity.append(current_val)

    # Close any open position at last price
    if position:
        last_price = df.iloc[-1]["close"]
        unreal_pnl = position["shares"] * last_price - position["cost"]
        trades.append({
            "ticker": symbol, "date": str(df.iloc[-1]["date"]),
            "action": "OPEN (unrealized)",
            "price": round(last_price, 2),
            "shares": round(position["shares"], 4),
            "pnl": round(unreal_pnl, 2),
            "reason": "Position open at period end",
            "capital": round(cash + position["shares"] * last_price, 2)
        })
        equity.append(cash + position["shares"] * last_price)

    # Summary stats
    final_cap  = equity[-1]
    sells      = [t for t in trades if t["action"] == "SELL"]
    wins       = [t for t in sells if t["pnl"] > 0]
    open_pos   = [t for t in trades if "OPEN" in t["action"]]
    total_pnl  = sum(t["pnl"] for t in sells) + sum(t["pnl"] for t in open_pos)
    equity_arr = np.array(equity)
    peaks      = np.maximum.accumulate(equity_arr)
    max_dd     = float(np.max((peaks - equity_arr) / peaks) * 100)

    return {
        "ticker":       symbol,
        "total_trades": len(sells),
        "win_rate":     round(len(wins) / len(sells) * 100) if sells else 0,
        "total_pnl":    round(total_pnl, 2),
        "total_return": round((final_cap - STARTING_CAP) / STARTING_CAP * 100, 2),
        "max_dd":       round(max_dd, 2),
        "final_cap":    round(final_cap, 2),
        "equity":       equity,
        "trades":       trades
    }

print("Running backtests...")
results = {}
all_trades = []
for t, df in processed.items():
    results[t] = backtest(df, t)
    all_trades.extend(results[t]["trades"])
    r = results[t]
    print(f"  {t}: {r['total_return']:+.1f}% | {r['total_trades']} trades | {r['win_rate']}% win | DD {r['max_dd']:.1f}%")
print("\nDone.")

In [ ]:
# ── Cell 7: Leaderboard table ──────────────────────────────────────────────────
from tabulate import tabulate

rows = []
for t, r in sorted(results.items(), key=lambda x: x[1]["total_return"], reverse=True):
    rows.append([
        t,
        f"{r['total_return']:+.2f}%",
        f"${r['total_pnl']:+.0f}",
        r['total_trades'],
        f"{r['win_rate']}%",
        f"{r['max_dd']:.1f}%",
        f"${r['final_cap']:,.0f}"
    ])

print(tabulate(
    rows,
    headers=["Ticker", "Return", "P&L", "Trades", "Win Rate", "Max DD", "Final Capital"],
    tablefmt="rounded_outline"
))

In [ ]:
# ── Cell 8: Trade log ─────────────────────────────────────────────────────────
trades_df = pd.DataFrame(all_trades)
trades_df["pnl"] = trades_df["pnl"].fillna("-")
print(trades_df[["ticker","date","action","price","shares","pnl","reason"]].to_string(index=False))

In [ ]:
# ── Cell 9: Equity curves chart ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: Equity curves
ax1 = axes[0]
colors = ["#1D9E75","#378ADD","#D85A30","#7F77DD","#BA7517","#3C3489","#639922"]
for i, (t, r) in enumerate(results.items()):
    ax1.plot(r["equity"], label=f"{t} ({r['total_return']:+.1f}%)",
             color=colors[i % len(colors)], linewidth=1.5)
ax1.axhline(STARTING_CAP, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax1.set_title(f"Equity Curves — {LOOKBACK_DAYS}d backtest · SMA{SMA_PERIOD}+RSI+Volume · ${STARTING_CAP:,} starting capital",
              fontsize=12, pad=10)
ax1.set_ylabel("Portfolio Value ($)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.legend(fontsize=9, ncol=4)
ax1.grid(axis="y", alpha=0.3)
ax1.spines[["top","right"]].set_visible(False)

# Bottom: Return bar chart
ax2 = axes[1]
tickers_sorted = sorted(results.keys(), key=lambda t: results[t]["total_return"], reverse=True)
returns = [results[t]["total_return"] for t in tickers_sorted]
bar_colors = ["#1D9E75" if r >= 0 else "#D85A30" for r in returns]
bars = ax2.bar(tickers_sorted, returns, color=bar_colors, width=0.5, zorder=3)
for bar, val in zip(bars, returns):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.2 if val >= 0 else -0.5),
             f"{val:+.1f}%", ha="center", va="bottom" if val >= 0 else "top", fontsize=10, fontweight="bold")
ax2.axhline(0, color="gray", linewidth=0.8)
ax2.set_title("Total Return by Ticker", fontsize=12, pad=10)
ax2.set_ylabel("Return (%)")
ax2.grid(axis="y", alpha=0.3, zorder=0)
ax2.spines[["top","right"]].set_visible(False)

plt.tight_layout(pad=3)
plt.savefig("backtest_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as backtest_results.png")

In [ ]:
# ── Cell 10: Parameter sensitivity test ───────────────────────────────────────
# Tests how sensitive NVDA results are to stop loss and RSI settings
print("Parameter sensitivity on NVDA\n")
nvda_df = processed.get("NVDA")
if nvda_df is None:
    print("NVDA data not available")
else:
    sensitivity_rows = []
    for sl in [2, 3, 4, 5, 6]:
        for rsi_max in [60, 65, 70]:
            orig_sl  = STOP_LOSS_PCT
            orig_rsi = RSI_BUY_MAX
            STOP_LOSS_PCT = sl / 100
            RSI_BUY_MAX   = rsi_max
            r = backtest(nvda_df, "NVDA")
            sensitivity_rows.append([f"{sl}%", rsi_max,
                f"{r['total_return']:+.1f}%", r['total_trades'],
                f"{r['win_rate']}%", f"{r['max_dd']:.1f}%"])
            STOP_LOSS_PCT = orig_sl
            RSI_BUY_MAX   = orig_rsi

    print(tabulate(
        sensitivity_rows,
        headers=["Stop Loss", "RSI Max", "Return", "Trades", "Win Rate", "Max DD"],
        tablefmt="rounded_outline"
    ))

In [ ]:
# ── Cell 11: Export results to CSV ────────────────────────────────────────────
trades_df.to_csv("trade_log.csv", index=False)

summary_df = pd.DataFrame([
    {"ticker": t, "return_pct": r["total_return"], "pnl": r["total_pnl"],
     "trades": r["total_trades"], "win_rate": r["win_rate"],
     "max_drawdown": r["max_dd"], "final_capital": r["final_cap"]}
    for t, r in results.items()
]).sort_values("return_pct", ascending=False)

summary_df.to_csv("backtest_summary.csv", index=False)
print("Exported: trade_log.csv and backtest_summary.csv")
print("\nAll done. Check the Files panel on the left to download results.")